In [1]:
import numpy as np
import math
import matplotlib.pyplot as plt

### stress rotation
The initial stress profile we are using is based on mini-frac test measurements conducted in boreholes TM1750 to TM2250. The vertical stress (Sv) is assumed to be lithostatic, with a magnitude of 26.5 MPa. The horizontal stresses are defined as Sh = 0.55Sv and SH = 0.96Sv, with the direction of SHmax oriented approximately N112°E (Bröker and Ma, 2022; Bröker et al., 2024). 

we need to rotate the stress tensor from the observations, because the stress directions are in north-east geological coords instead of the mesh direcitons

#### ratation by steps

In [2]:
# Principal stresses
sigma1 = -25.4
sigma2 = -14.6
sigma3 = -26.5

# Angle in degrees
theta_deg = 22 # 112 - 90
theta_rad = np.radians(theta_deg)

# Principal directions (unit vectors)
v1 = [np.cos(theta_rad), -np.sin(theta_rad), 0]
v2 = [np.sin(theta_rad),  np.cos(theta_rad), 0]
v3 = [0, 0, 1]

# Rotation matrix: columns are principal directions
R = np.column_stack((v1, v2, v3))

# Diagonal principal stress tensor
sigma_principal = np.diag([sigma1, sigma2, sigma3])

# Transform to original coordinate system
sigma_original = R @ sigma_principal @ R.T

# Display the stress tensor
# np.set_printoptions(precision=2, suppress=True)
print("Stress tensor in original coordinate system:")
print(sigma_original)


Stress tensor in original coordinate system:
[[-23.88443492   3.7511552    0.        ]
 [  3.7511552  -16.11556508   0.        ]
 [  0.           0.         -26.5       ]]


In [3]:
f_rot = 1.1539 # in rads, the rotation for aligning fault to y axis (fault strike N66E)

rotation_angle = f_rot

Rz = np.asmatrix([
    [np.cos(rotation_angle), -np.sin(rotation_angle), 0],
    [np.sin(rotation_angle),  np.cos(rotation_angle), 0],
    [0, 0, 1]
])

rotated_sigma = Rz @ sigma_original @ Rz.T
rotated_sigma

matrix([[-20.16705581,  -5.39741534,   0.        ],
        [ -5.39741534, -19.83294419,   0.        ],
        [  0.        ,   0.        , -26.5       ]])

In [4]:
# just doucle check
eigenvalues, eigenvectors = np.linalg.eig(rotated_sigma)

print("Eigenvalues:")
print(eigenvalues)
print("\nEigenvectors:")
print(eigenvectors[:,0])

t = eigenvectors[:,0]
90 - np.degrees(np.arctan(t[1]/t[0])) # should be ~ 46

Eigenvalues:
[-25.4 -14.6 -26.5]

Eigenvectors:
[[-0.71796109]
 [-0.69608323]
 [ 0.        ]]


matrix([[45.88640002]])

In [5]:
rho = 2620 # Water-saturated bulk density
g = -9.81

# if the measurements are from the tunnel, the height differece is 15.678 m to the origin point in my mesh
# 1485.38 (lab origin) - 1469.702 (point in BFE5 that I used as the origin point)
# (sigma_v - sigma_v0) / (z-z0)= rho * g   
# sigma_v0 = sigma_v - rho * g * z
sigma_v_o = (rotated_sigma[2,2] *1e6 - rho * g * (-15.678))
sigma_v_o/1e6 # at the origin point

-26.9029590916

In [6]:
# essentially fix the ratio sigma_h/sigma_v
ratio = rotated_sigma/rotated_sigma[2,2]
ratio

matrix([[ 0.76102097,  0.20367605, -0.        ],
        [ 0.20367605,  0.74841299, -0.        ],
        [-0.        , -0.        ,  1.        ]])

In [7]:
# calculate the gradients for all the stresses
grad_stress = - rho * g * ratio
grad_stress

matrix([[19559.91327609,  5234.92258723,     0.        ],
        [ 5234.92258723, 19235.86030881,     0.        ],
        [    0.        ,     0.        , 25702.2       ]])

In [8]:
# the stress tensor at the mesh origin point
simga_coor = ratio * sigma_v_o /1e6
simga_coor

matrix([[-20.47371613,  -5.47948846,   0.        ],
        [ -5.47948846, -20.13452401,   0.        ],
        [  0.        ,   0.        , -26.90295909]])